[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_13/2_sql.ipynb)

# Day 13: SQL (medium)

**How to work:** Read the tables first (columns, one row = what?). Write the query in your head in the order FROM, WHERE, GROUP BY, HAVING, SELECT, ORDER BY. Check the result for duplicates and NULLs before you open the solution.

**Today (31 min):**

| # | Question | Level | Minutes |
|---|---|---|---|
| Q1 | Year to date, month by month | medium | 6 |
| Q2 | A smoother DAU line for March | medium | 6 |
| Q3 | Joining the 1,000 club | medium | 7 |
| Q4 | How concentrated is revenue? | medium | 7 |
| Q5 | Week over week signups (review) | medium | 5 |

**Handbook:** [SQL](https://mahsa-agz.github.io/ds-handbook/#sql), [cheat sheets](https://mahsa-agz.github.io/ds-handbook/#cheat)

## Setup

Run this cell once before the questions.

In [ ]:
import sqlite3
import pandas as pd

def run(q, con=None):
    """Runs a query and shows the result as a table (SELECT / WITH); other statements just run."""
    con = con or db
    if not q.strip():
        print("Write your query between the triple quotes, then run the cell.")
        return None
    if q.strip().lower().startswith(("select", "with")):
        return pd.read_sql_query(q, con)
    con.executescript(q)
    print("done")

# Finds the data folder (local copy of the repository, or /content/data in Colab) and downloads a missing file
# from the ds-handbook repository on GitHub (or from its original source).
import os, sys, urllib.request

RAW = "https://raw.githubusercontent.com/mahsa-agz/mahsa-agz.github.io/main/ds-handbook/exam_prep/data/"

def _find_data():
    if "google.colab" in sys.modules:
        os.makedirs("/content/data", exist_ok=True)
        return "/content/data"
    for p in ("../data", "data", "exam_prep/data"):
        if os.path.isdir(p):
            return p
    os.makedirs("data", exist_ok=True)
    return "data"

DATA = _find_data()
print("data folder:", DATA)

def data_path(name, url=None):
    """Local path of a data file; downloads it from the repository (or from url) if it is not there yet."""
    path = os.path.join(DATA, name)
    if not os.path.exists(path):
        for source in (RAW + name, url):
            if not source:
                continue
            try:
                print("downloading", name)
                urllib.request.urlretrieve(source, path)
                break
            except Exception as e:
                print("  could not download from", source, "->", e)
    return path

db = sqlite3.connect(':memory:')

# Made-up shopping-app tables (generated with a fixed seed, so everyone gets the same rows).
import numpy as np

def make_app_tables(con, seed=7):
    rs = np.random.RandomState(seed)
    start, end = pd.Timestamp("2024-01-01"), pd.Timestamp("2024-03-31")
    n = 1500
    countries, c_p = ["US", "KR", "BR", "ID", "DE"], [0.30, 0.25, 0.20, 0.15, 0.10]
    platforms, p_p = ["ios", "android", "web"], [0.40, 0.45, 0.15]
    channels, ch_p = ["organic", "paid_social", "search", "referral"], [0.35, 0.30, 0.20, 0.15]
    signup_off = np.sort(rs.randint(0, 60, n))                     # day offsets 0..59 (Jan 1 to Feb 29)
    country = rs.choice(len(countries), n, p=c_p)
    platform = rs.choice(len(platforms), n, p=p_p)
    channel = rs.choice(len(channels), n, p=ch_p)
    referred_by = [None] * n
    for i in range(n):
        if channels[channel[i]] == "referral":
            earlier = np.nonzero(signup_off[:i] < signup_off[i])[0]
            if len(earlier):
                referred_by[i] = int(earlier[rs.randint(len(earlier))]) + 1
            else:
                channel[i] = 0                                       # nobody to refer them: organic
    users = pd.DataFrame({
        "user_id": np.arange(1, n + 1),
        "signup_date": [(start + pd.Timedelta(days=int(d))).strftime("%Y-%m-%d") for d in signup_off],
        "country": [countries[c] for c in country],
        "platform": [platforms[p] for p in platform],
        "channel": [channels[c] for c in channel],
        "referred_by": pd.array(referred_by, dtype="Int64"),
    })
    plat_boost = {"ios": 0.08, "android": 0.0, "web": -0.10}
    chan_boost = {"organic": 0.05, "paid_social": -0.08, "search": 0.0, "referral": 0.08}
    act, ev, buys = [], [], []
    total_days = (end - start).days + 1
    for i in range(n):
        uid = i + 1
        p = rs.beta(2, 3) + plat_boost[platforms[platform[i]]] + chan_boost[channels[channel[i]]]
        p = min(max(p, 0.03), 0.9)
        life = 1 if rs.rand() < 0.30 else 1 + int(rs.exponential(35))   # days until the user stops coming back
        buyer = rs.rand() < 0.55
        prev = True
        for t in range(0, total_days - signup_off[i]):
            if t == 0:
                active = True
            elif t >= life:
                active = rs.rand() < 0.01                                 # rare comeback after churn
            else:
                active = rs.rand() < (min(p + 0.25, 0.95) if prev else p * 0.7)
            prev = active
            if not active:
                continue
            day = start + pd.Timedelta(days=int(signup_off[i] + t))
            sessions = 1 + rs.poisson(0.8)
            minutes = int(sum(rs.randint(2, 25) for _ in range(sessions)))
            act.append((uid, day.strftime("%Y-%m-%d"), sessions, minutes))
            clock = day + pd.Timedelta(seconds=int(rs.randint(6 * 3600, 22 * 3600)))
            if rs.rand() < 0.65:
                steps = ["view_item"]
                if rs.rand() < 0.40:
                    steps.append("add_to_cart")
                    if rs.rand() < 0.55:
                        steps.append("checkout")
                        if buyer and rs.rand() < 0.75:
                            steps.append("purchase")
                elif rs.rand() < 0.03:
                    steps = ["add_to_cart"]                               # from the wishlist, no view that day
                for s in steps:
                    clock = clock + pd.Timedelta(seconds=int(rs.randint(15, 900)))
                    ev.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), s))
                    if s == "purchase":
                        amount = round(float(rs.gamma(2.0, 18.0)) + 5, 2)
                        status = "refunded" if rs.rand() < 0.07 else "completed"
                        buys.append((uid, clock.strftime("%Y-%m-%d %H:%M:%S"), amount, status))
    activity = pd.DataFrame(act, columns=["user_id", "activity_date", "sessions", "minutes"])
    events = pd.DataFrame(ev, columns=["user_id", "event_time", "event_name"])
    events = events.sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    events.insert(0, "event_id", np.arange(1, len(events) + 1))
    purchases = pd.DataFrame(buys, columns=["user_id", "order_time", "amount", "status"])
    purchases = purchases.sort_values(["order_time", "user_id"], kind="mergesort").reset_index(drop=True)
    purchases.insert(0, "order_id", np.arange(1, len(purchases) + 1))
    # raw event log as it arrives from the apps: retries create exact copies, double taps create near copies
    raw = events.copy()
    dup = raw[rs.rand(len(raw)) < 0.04]
    tap = raw[rs.rand(len(raw)) < 0.01].copy()
    tap["event_time"] = (pd.to_datetime(tap["event_time"]) +
                         pd.to_timedelta(rs.randint(1, 4, len(tap)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    raw = pd.concat([raw, dup, tap]).sort_values(["event_time", "user_id"], kind="mergesort").reset_index(drop=True)
    raw = raw.drop(columns="event_id")
    raw.insert(0, "raw_id", np.arange(1, len(raw) + 1))
    raw["received_at"] = (pd.to_datetime(raw["event_time"]) +
                          pd.to_timedelta(rs.randint(1, 600, len(raw)), unit="s")).dt.strftime("%Y-%m-%d %H:%M:%S")
    for name, df in [("app_users", users), ("daily_activity", activity), ("app_events", events),
                     ("purchases", purchases), ("app_events_raw", raw)]:
        df.to_sql(name, con, index=False, if_exists="replace")

make_app_tables(db)

# Real data: Northwind orders (enlarged version, 2012 to 2023). Copied into tables with an nw_ prefix.
db.execute("ATTACH DATABASE ? AS nw", (data_path("northwind.db", "https://raw.githubusercontent.com/jpwhite3/northwind-SQLite3/main/dist/northwind.db"),))
db.executescript("""
CREATE TABLE nw_orders AS SELECT OrderID AS order_id, CustomerID AS customer_id, EmployeeID AS employee_id,
       substr(OrderDate, 1, 10) AS order_date, ShipCountry AS ship_country FROM nw.Orders;
CREATE TABLE nw_order_details AS SELECT OrderID AS order_id, ProductID AS product_id, UnitPrice AS unit_price,
       Quantity AS quantity, Discount AS discount FROM nw."Order Details";
CREATE TABLE nw_products AS SELECT ProductID AS product_id, ProductName AS product_name, CategoryID AS category_id
       FROM nw.Products;
CREATE TABLE nw_categories AS SELECT CategoryID AS category_id, CategoryName AS category_name FROM nw.Categories;
""")
db.execute("DETACH DATABASE nw")

# Real data: UCI Online Retail (UK online shop, 2010-12 to 2011-12; CC BY 4.0).
def online_retail():
    path = os.path.join(DATA, "online_retail.csv.gz")
    if not os.path.exists(path):
        import io, zipfile
        print("downloading Online Retail (about 23 MB, then converting the xlsx)")
        z = zipfile.ZipFile(io.BytesIO(urllib.request.urlopen("https://archive.ics.uci.edu/static/public/352/online+retail.zip").read()))
        pd.read_excel(io.BytesIO(z.read("Online Retail.xlsx"))).to_csv(path, index=False, compression="gzip")
    return pd.read_csv(path, dtype={"InvoiceNo": str, "StockCode": str})

_o = online_retail()
_o.columns = ["invoice_no", "stock_code", "description", "quantity", "invoice_date", "unit_price", "customer_id",
              "country"]
_o["customer_id"] = _o["customer_id"].astype("Int64")
_o["invoice_date"] = pd.to_datetime(_o["invoice_date"]).dt.strftime("%Y-%m-%d %H:%M:%S")
_o.to_sql("retail", db, index=False)

print("tables:", ", ".join(r[0] for r in db.execute("SELECT name FROM sqlite_master WHERE type = 'table' ORDER BY name")))

## Tables

**Made-up shopping-app tables** (invented for practice, generated with a fixed seed: 1,500 users signed up
2024-01-01 to 2024-02-29, activity until 2024-03-31):

| Table | One row is | Columns |
|---|---|---|
| `app_users` | one user | `user_id`, `signup_date` ('YYYY-MM-DD'), `country` (US/KR/BR/ID/DE), `platform` (ios/android/web), `channel` (organic/paid_social/search/referral), `referred_by` (user_id of the referrer, NULL unless channel = referral) |
| `daily_activity` | one user on one day they opened the app | `user_id`, `activity_date` ('YYYY-MM-DD'), `sessions`, `minutes`. No row = not active that day |
| `app_events` | one shopping event | `event_id`, `user_id`, `event_time` ('YYYY-MM-DD HH:MM:SS'), `event_name` (view_item / add_to_cart / checkout / purchase) |
| `purchases` | one order | `order_id`, `user_id`, `order_time`, `amount` (USD), `status` (completed / refunded) |
| `app_events_raw` | one event as received from the apps (has retries and double taps) | `raw_id`, `user_id`, `event_time`, `event_name`, `received_at` |

**Real tables: Northwind** (orders of a food wholesaler; enlarged build with dates 2012 to 2023; MIT license,
jpwhite3/northwind-SQLite3). Columns renamed to snake_case:

| Table | One row is | Columns |
|---|---|---|
| `nw_orders` | one order (16,282) | `order_id`, `customer_id`, `employee_id`, `order_date` ('YYYY-MM-DD'), `ship_country` |
| `nw_order_details` | one product line in an order (609,283) | `order_id`, `product_id`, `unit_price`, `quantity`, `discount` (0 to 1). Revenue = `unit_price * quantity * (1 - discount)` |
| `nw_products` / `nw_categories` | one product (77) / category (8) | `product_id`, `product_name`, `category_id` / `category_id`, `category_name` |

**Real table: UCI Online Retail** (a UK online gift shop, 2010-12-01 to 2011-12-09; CC BY 4.0):

| Table | One row is | Columns |
|---|---|---|
| `retail` | one product line of an invoice (541,909) | `invoice_no` (starts with 'C' = cancellation), `stock_code`, `description`, `quantity` (negative for returns), `invoice_date` ('YYYY-MM-DD HH:MM:SS'), `unit_price`, `customer_id` (NULL for about 25% of rows), `country` |

The queries run in SQLite. They are written in standard SQL that also works in PostgreSQL; differences (dates, integer division) are noted in the solutions.

---
### Q1. Year to date, month by month

*medium, about 6 min*

Northwind (real schema). For **2021 and 2022**, return one row per month with `month` ('YYYY-MM'),
`revenue` (that month), `ytd` (revenue from January of the same year up to and including this month) and
`pct_of_year` (ytd as a percent of the full year's revenue, 1 decimal). Round money to whole units.
Revenue of a line is `unit_price * quantity * (1 - discount)`.

Example: if January is 100 and February 80 in a year that totals 1,000, then February has `ytd = 180` and
`pct_of_year = 18.0`. January of the next year starts again from its own revenue.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "cumulative from the start of the year, restart every year". Pattern: running total `SUM(...) OVER (PARTITION BY year ORDER BY month)`, plus a plain `SUM(...) OVER (PARTITION BY year)` for the total.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE monthly: year, month, revenue (join orders and order details, filter the two years).
2. `SUM(revenue) OVER (PARTITION BY yr ORDER BY month ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)`.
3. Divide by `SUM(revenue) OVER (PARTITION BY yr)` (no ORDER BY = whole partition).

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH monthly AS (
    SELECT substr(o.order_date, 1, 4) AS yr,
           substr(o.order_date, 1, 7) AS month,
           SUM(d.unit_price * d.quantity * (1 - d.discount)) AS revenue
    FROM nw_orders o
    JOIN nw_order_details d ON d.order_id = o.order_id
    WHERE o.order_date >= '2021-01-01' AND o.order_date < '2023-01-01'
    GROUP BY substr(o.order_date, 1, 4), substr(o.order_date, 1, 7)
)
SELECT month,
       ROUND(revenue, 0) AS revenue,
       ROUND(SUM(revenue) OVER (PARTITION BY yr ORDER BY month
                                ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW), 0) AS ytd,
       ROUND(100.0 * SUM(revenue) OVER (PARTITION BY yr ORDER BY month
                                        ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW)
             / SUM(revenue) OVER (PARTITION BY yr), 1) AS pct_of_year
FROM monthly
ORDER BY month
```

Result:

| month | revenue | ytd | pct_of_year |
|---|---|---|---|
| 2021-01 | 3474864.0 | 3474864.0 | 8.4 |
| 2021-02 | 2953520.0 | 6428384.0 | 15.5 |
| 2021-03 | 3434464.0 | 9862848.0 | 23.8 |
| 2021-04 | 3255793.0 | 13118641.0 | 31.7 |
| 2021-05 | 3319861.0 | 16438502.0 | 39.7 |
| 2021-06 | 3510713.0 | 19949215.0 | 48.2 |
| 2021-07 | 3917400.0 | 23866615.0 | 57.7 |
| 2021-08 | 3443530.0 | 27310145.0 | 66.0 |
| 2021-09 | 3186946.0 | 30497091.0 | 73.7 |
| 2021-10 | 3071773.0 | 33568864.0 | 81.2 |
| 2021-11 | 3408890.0 | 36977754.0 | 89.4 |
| 2021-12 | 4377795.0 | 41355550.0 | 100.0 |

(24 rows, first 12 shown)

**Why:** With ORDER BY inside OVER, SUM becomes cumulative; PARTITION BY yr restarts it every January. Without ORDER BY, the same SUM covers the whole partition, which gives the yearly total on every row. 2021 reaches 81.2% of its revenue by October, 2022 reaches 82.3%. Writing the frame `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW` explicitly matters: the default frame is RANGE, which treats rows with the same ORDER BY value as one block (no difference here because months are unique, but it bites when the order key has duplicates).

**Complexity:** Join and group by over the order lines of two years, then windows over 24 rows.

**Common mistakes:** Forgetting PARTITION BY yr (ytd keeps growing into the next year). Using the default RANGE frame with duplicate order keys (all tied rows get the same running total). Dividing by the running total instead of the year total.

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q2. A smoother DAU line for March

*medium, about 6 min*

Shopping app (made up). Daily active users (DAU) = number of users with a row in `daily_activity`
that day. The dashboard for **March 2024** needs `activity_date`, `dau`, `dau_7d` (average DAU of this day and the
6 days before, 1 decimal) and `n_days` (how many days went into the average).

Important: the 7-day average on 2024-03-01 must use Feb 24 to Mar 1, so `n_days` must be 7 on every March row.

Follow-up: what breaks if some dates have no activity rows at all, and how do you fix it?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "average of the last 7 days on every row". Pattern: moving window `AVG(...) OVER (ORDER BY day ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)`. Trap: when is the March filter applied?

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE dau: COUNT(*) per activity_date over ALL dates.
2. CTE smooth: add the moving AVG and a moving COUNT(*) with the same frame.
3. Only now filter `activity_date >= '2024-03-01'` in the outer query.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH dau AS (
    SELECT activity_date, COUNT(*) AS dau
    FROM daily_activity
    GROUP BY activity_date
), smooth AS (
    SELECT activity_date, dau,
           ROUND(AVG(dau) OVER (ORDER BY activity_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW), 1) AS dau_7d,
           COUNT(*) OVER (ORDER BY activity_date ROWS BETWEEN 6 PRECEDING AND CURRENT ROW)             AS n_days
    FROM dau
)
SELECT activity_date, dau, dau_7d, n_days
FROM smooth
WHERE activity_date >= '2024-03-01' AND activity_date < '2024-04-01'
ORDER BY activity_date
```

Result:

| activity_date | dau | dau_7d | n_days |
|---|---|---|---|
| 2024-03-01 | 260 | 281.4 | 7 |
| 2024-03-02 | 256 | 277.4 | 7 |
| 2024-03-03 | 253 | 272.6 | 7 |
| 2024-03-04 | 236 | 264.0 | 7 |
| 2024-03-05 | 239 | 256.6 | 7 |
| 2024-03-06 | 227 | 249.1 | 7 |
| 2024-03-07 | 216 | 241.0 | 7 |
| 2024-03-08 | 214 | 234.4 | 7 |
| 2024-03-09 | 194 | 225.6 | 7 |
| 2024-03-10 | 197 | 217.6 | 7 |
| 2024-03-11 | 191 | 211.1 | 7 |
| 2024-03-12 | 198 | 205.3 | 7 |

(31 rows, first 12 shown)

**Why:** WHERE runs before window functions. If you put the March filter in the same SELECT as the AVG, the window only sees March rows and the first 6 days average fewer than 7 days (n_days 1, 2, ... 6). Computing the window in a CTE and filtering outside keeps February in the frame. The smooth line shows DAU falling through March (signups stopped at the end of February). Follow-up: ROWS counts rows, not days. If a date has no rows, the frame reaches 8 or more calendar days back. Fix: build a calendar (recursive CTE or `generate_series` in PostgreSQL), LEFT JOIN the counts and COALESCE to 0, or in PostgreSQL use `RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW` on a date column.

**Complexity:** One group by, then a window over 91 daily rows.

**Common mistakes:** Filtering the month before computing the window. `ROWS BETWEEN 7 PRECEDING` (that is 8 days). Averaging user-level rows instead of daily counts. Assuming no gaps in the dates.

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-query-order](https://mahsa-agz.github.io/ds-handbook/#sql-query-order), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
### Q3. Joining the 1,000 club

*medium, about 7 min*

Online Retail (real). A customer joins the **1,000 club** on the first day their **cumulative net
spend** (all their lines up to and including that day, `quantity * unit_price`, cancellations included as negative)
reaches **1,000 or more**. Ignore rows without `customer_id`.

Return, per month ('YYYY-MM'), `new_1000_club` (customers who joined that month) and `club_size` (members so far,
cumulative over months).

Example: a customer with daily spend 600 (Jan 5), -200 (Jan 9, a return), 700 (Feb 2) reaches 600, 400, 1,100,
so they join on Feb 2. Note that a return can push a customer back below 1,000; they still joined on the first day.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "the first day a cumulative sum crosses a threshold". Pattern: running total per customer, then MIN(day) where the running total >= 1000; a second running total over months.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE daily: spend per customer per day.
2. CTE cum: `SUM(spend) OVER (PARTITION BY customer_id ORDER BY day ROWS UNBOUNDED PRECEDING)`.
3. CTE firsts: `MIN(day)` per customer WHERE cum_spend >= 1000.
4. Group firsts by month; `SUM(COUNT(*)) OVER (ORDER BY month)` for club_size.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH daily AS (
    SELECT customer_id, substr(invoice_date, 1, 10) AS day, SUM(quantity * unit_price) AS spend
    FROM retail
    WHERE customer_id IS NOT NULL
    GROUP BY customer_id, substr(invoice_date, 1, 10)
), cum AS (
    SELECT customer_id, day,
           SUM(spend) OVER (PARTITION BY customer_id ORDER BY day ROWS UNBOUNDED PRECEDING) AS cum_spend
    FROM daily
), firsts AS (
    SELECT customer_id, MIN(day) AS joined_day
    FROM cum
    WHERE cum_spend >= 1000
    GROUP BY customer_id
)
SELECT substr(joined_day, 1, 7) AS month,
       COUNT(*) AS new_1000_club,
       SUM(COUNT(*)) OVER (ORDER BY substr(joined_day, 1, 7)) AS club_size
FROM firsts
GROUP BY substr(joined_day, 1, 7)
ORDER BY month
```

Result:

| month | new_1000_club | club_size |
|---|---|---|
| 2010-12 | 111 | 111 |
| 2011-01 | 110 | 221 |
| 2011-02 | 81 | 302 |
| 2011-03 | 149 | 451 |
| 2011-04 | 111 | 562 |
| 2011-05 | 147 | 709 |
| 2011-06 | 125 | 834 |
| 2011-07 | 103 | 937 |
| 2011-08 | 111 | 1048 |
| 2011-09 | 163 | 1211 |
| 2011-10 | 180 | 1391 |
| 2011-11 | 210 | 1601 |

(13 rows, first 12 shown)

**Why:** Aggregate to one row per customer and day first, so the running total moves once per day. `ROWS UNBOUNDED PRECEDING` is short for `ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW`. Taking MIN(day) among the rows that pass the threshold gives the first crossing even if the customer later drops below. `SUM(COUNT(*)) OVER (...)` is a window over an aggregate: GROUP BY runs first, then the window adds up the monthly counts. December 2011 is small because the data ends on 2011-12-09.

**Complexity:** Group by over the lines, one window sort per customer, then small group bys.

**Common mistakes:** Running the window over raw lines (several rows per day; you still get the right first day, but work on 400k rows instead of about 18k). Using `MAX` instead of `MIN` for the crossing day. Forgetting that people who returned items can drop back below the threshold (they must not be counted twice).

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-cte](https://mahsa-agz.github.io/ds-handbook/#sql-cte)

</details>

---
### Q4. How concentrated is revenue?

*medium, about 7 min*

Shopping app (made up). Sort buyers by their **completed** revenue, largest first (ties: smaller
user_id first). What is the **smallest number of buyers** whose combined revenue is at least **80%** of all
completed revenue?

Return one row: `buyers_for_80pct`, `n_buyers` (users with at least one completed order), `pct_of_buyers`
(1 decimal) and `n_users` (all users in app_users).

Example: revenues 50, 30, 10, 10 (total 100). Cumulative shares are 50%, 80%, 90%, 100%, so 2 buyers out of 4
(50.0%) make 80% of revenue.

Follow-up: a PM says "half of our buyers make 80% of revenue, so revenue is not concentrated". What else would you
show?

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "cumulative share of a total, sorted". Pattern: running total over `ORDER BY revenue DESC` divided by the grand total `SUM(revenue) OVER ()`; the answer is the first row that reaches 0.8.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE: revenue per user (completed only).
2. CTE: ROW_NUMBER by revenue desc, `COUNT(*) OVER ()`, and cum_share = running SUM / `SUM(...) OVER ()`.
3. `MIN(rnk) WHERE cum_share >= 0.8`.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH spend AS (
    SELECT user_id, SUM(amount) AS revenue
    FROM purchases
    WHERE status = 'completed'
    GROUP BY user_id
), cum AS (
    SELECT user_id, revenue,
           ROW_NUMBER() OVER (ORDER BY revenue DESC, user_id) AS rnk,
           COUNT(*) OVER () AS n_buyers,
           SUM(revenue) OVER (ORDER BY revenue DESC, user_id ROWS UNBOUNDED PRECEDING)
             / SUM(revenue) OVER () AS cum_share
    FROM spend
)
SELECT MIN(rnk) AS buyers_for_80pct,
       MAX(n_buyers) AS n_buyers,
       ROUND(100.0 * MIN(rnk) / MAX(n_buyers), 1) AS pct_of_buyers,
       (SELECT COUNT(*) FROM app_users) AS n_users
FROM cum
WHERE cum_share >= 0.8
```

Result:

| buyers_for_80pct | n_buyers | pct_of_buyers | n_users |
|---|---|---|---|
| 203 | 403 | 50.4 | 1500 |

**Why:** `SUM(...) OVER ()` with an empty window is the grand total on every row, so the ratio is the cumulative share. The tie breaker in both ORDER BYs keeps ROW_NUMBER and the running sum in the same order. About half of the 403 buyers make 80% of revenue, which is less concentrated than the classic 80/20. Follow-up: the base matters. Only 403 of 1,500 users buy at all, so relative to all users about 13.5% make 80% of revenue. Show both, plus a Lorenz curve or the top 10% share, and how it changes over time.

**Complexity:** Group by, then windows over about 400 rows.

**Common mistakes:** Running sum without the tie breaker (equal revenues get the same RANGE sum and the count jumps). Using `> 0.8` and missing the row that is exactly 80%. Including refunded orders.

**Learn more:** [sql-running-totals](https://mahsa-agz.github.io/ds-handbook/#sql-running-totals), [sql-ratios](https://mahsa-agz.github.io/ds-handbook/#sql-ratios)

</details>

---
### Q5. Week over week signups (review)

*medium, about 5 min*

Shopping app (made up). Weeks start on **Monday**. Return `week_start`, `signups`, `prev_week` and
`wow_pct` (week over week percent change, 1 decimal) for every signup week.

In SQLite, the Monday of the week of date d is `date(d, 'weekday 0', '-6 days')` (move forward to Sunday, then
back 6 days). Example: 2024-01-03 (a Wednesday) belongs to week 2024-01-01.

Then explain the last row.

In [ ]:
run("""

""")

<details><summary><b>Hint 1</b></summary>

Signal: "this week versus last week". Pattern: truncate dates to the week, aggregate, then LAG.

</details>

<details><summary><b>Hint 2</b></summary>

1. CTE weekly: week_start and COUNT(*).
2. `LAG(signups) OVER (ORDER BY week_start)`.
3. Percent change with 100.0 to avoid integer division.

</details>

<details><summary><b>Solution</b></summary>

```sql
WITH weekly AS (
    SELECT date(signup_date, 'weekday 0', '-6 days') AS week_start,
           COUNT(*) AS signups
    FROM app_users
    GROUP BY date(signup_date, 'weekday 0', '-6 days')
)
SELECT week_start,
       signups,
       LAG(signups) OVER (ORDER BY week_start) AS prev_week,
       ROUND(100.0 * (signups - LAG(signups) OVER (ORDER BY week_start))
             / LAG(signups) OVER (ORDER BY week_start), 1) AS wow_pct
FROM weekly
ORDER BY week_start
```

Result:

| week_start | signups | prev_week | wow_pct |
|---|---|---|---|
| 2024-01-01 | 174 | NULL | NULL |
| 2024-01-08 | 179 | 174 | 2.9 |
| 2024-01-15 | 170 | 179 | -5.0 |
| 2024-01-22 | 181 | 170 | 6.5 |
| 2024-01-29 | 175 | 181 | -3.3 |
| 2024-02-05 | 168 | 175 | -4.0 |
| 2024-02-12 | 200 | 168 | 19.0 |
| 2024-02-19 | 156 | 200 | -22.0 |
| 2024-02-26 | 97 | 156 | -37.8 |

**Why:** Truncating to the week start makes a stable group key; LAG then compares neighbouring weeks. The last week (2024-02-26) shows -37.8%, but signups in the data stop on 2024-02-29, so that week has only 4 days. Same lesson as the partial month on day 12: compare complete periods or per-day rates. PostgreSQL: `date_trunc('week', signup_date)` (weeks start on Monday there too).

**Complexity:** One group by and a window over 9 rows.

**Common mistakes:** Using `strftime('%W')` without the year (weeks of different years merge). Weeks starting on Sunday by accident. Reading the partial last week as a real drop.

**Learn more:** [sql-lag-lead](https://mahsa-agz.github.io/ds-handbook/#sql-lag-lead), [sql-dates](https://mahsa-agz.github.io/ds-handbook/#sql-dates)

</details>

---
## Redo list

Write down every question you got wrong, solved only after Hint 2, or finished over time. Solve it again from scratch 3 days and 7 days later.

**Next:** [Statistics, A/B testing and product](https://colab.research.google.com/github/mahsa-agz/mahsa-agz.github.io/blob/main/ds-handbook/exam_prep/day_13/3_stats.ipynb)